# 🏟️ 4 モデル日本語LLMベンチマーク選手権

LLM-jp Playground の 4 モデルに **公開された日本語LLMベンチマーク** を実走させ、メダルを争わせる。
玩具タスクではなく Nejumi LLM Leaderboard / llm-jp-eval / Open Japanese LLM Leaderboard で使われている標準ベンチマークの「ミニ実走版」。

## 競技種目

| # | 競技 | データセット | 形式 | 出典 |
|---|---|---|---|---|
| 1 | 🧠 常識推論 | `sbintuitions/JCommonsenseQA` | 5択 (0-4) | JGLUE [Kurihara+, NLP2022] |
| 2 | 📚 学術知識 | `nlp-waseda/JMMLU` | 4択 (A-D) | JMMLU [Yin+, 2024] |
| 3 | ➗ 数学CoT | `juletxara/mgsm` (`ja`) | 数値 | MGSM [Shi+, 2022] |
| 4 | 💬 自由対話 | 自前プロンプト | 相互審査 (Borda) | JaMT-Bench style |

## 採点

- 各競技: 1位=4点, 2位=3点, 3位=2点, 4位=1点 (Borda)
- 同点は同点扱い、点数の平均を分け合う
- 合計点で **総合王者**、メダル数 (🥇🥈🥉) も併記

## サンプルサイズ
- デフォルトは速さ優先 (各タスク 8〜12問、計 ~10 分)
- 各セルで `N_*` を増やせばより厳密な評価になる
- 本格的な評価は [llm-jp-eval](https://github.com/llm-jp/llm-jp-eval) で

## ライセンス
- **JCommonsenseQA**: CC BY-SA 4.0
- **JMMLU**: CC BY-NC-ND 4.0 (研究・評価目的のみ、商用不可)
- **MGSM**: CC BY-SA 4.0
- 出力データを再配布する場合は各ライセンスを尊重してください

エンドポイント: `https://llm-jp-playground.apps.llmc.nii.ac.jp/api/v1`


## 1. セットアップ

In [ ]:
import os, re, random, time, json, statistics, traceback
from collections import defaultdict
from openai import OpenAI
from IPython.display import display, Markdown
try:
    from datasets import load_dataset
except ImportError:
    print("⚠️ `datasets` が無いので入れます (Binder では requirements.txt 経由で入る想定)")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
    from datasets import load_dataset

client = OpenAI(
    base_url="https://llm-jp-playground.apps.llmc.nii.ac.jp/api/v1",
    api_key=os.environ.get("LLMJP_API_KEY", "dummy"),
    timeout=300.0,
)

def chat(model, prompt, system="日本語で。", max_tokens=2000, temperature=0.0,
         max_retries=2):
    """内部 streaming 1 ショット (03-06 と同じ系列)。
    ベンチマーク用なので temperature=0.0 がデフォルト (再現性重視)。
    返り値は (text, latency_sec, tokens_in_reasoning) のタプル。
    """
    t0 = time.time()
    sys_msg = system + "\n\n/no_think"
    msgs = [{"role": "system", "content": sys_msg},
            {"role": "user", "content": prompt}]
    extra = {"chat_template_kwargs": {"enable_thinking": False}}
    def _create(use_extra):
        kwargs = dict(model=model, messages=msgs, max_tokens=max_tokens,
                      temperature=temperature, stream=True)
        if use_extra:
            kwargs["extra_body"] = extra
        return client.chat.completions.create(**kwargs)
    last_reasoning = ""
    for attempt_i in range(max_retries + 1):
        try:
            try:
                stream = _create(True)
            except Exception:
                stream = _create(False)
        except Exception as e:
            if attempt_i < max_retries:
                time.sleep(1.0 + 0.5 * attempt_i); continue
            return "", time.time() - t0, 0
        content, reasoning = [], []
        interrupted = None
        try:
            for chunk in stream:
                if not chunk.choices: continue
                d = chunk.choices[0].delta
                c = getattr(d, "content", None)
                if c: content.append(c)
                for f in ("reasoning_content", "reasoning"):
                    v = getattr(d, f, None)
                    if v: reasoning.append(v); break
        except Exception as e:
            interrupted = type(e).__name__
        text = "".join(content).strip()
        last_reasoning = "".join(reasoning).strip()
        if text:
            return text, time.time() - t0, len(last_reasoning)
        if not interrupted:
            return last_reasoning, time.time() - t0, len(last_reasoning)
        if attempt_i < max_retries:
            time.sleep(1.5 ** attempt_i); continue
        return last_reasoning, time.time() - t0, len(last_reasoning)
    return last_reasoning, time.time() - t0, len(last_reasoning)

ALL = [m.id for m in client.models.list().data]
def pick(s):
    for m in ALL:
        if s.lower() in m.lower(): return m
    raise RuntimeError(f"no model matching {s!r}")

VOICES = {
    "🌸 LLM-jp 8b":  pick("llm-jp-4-8b"),
    "🗻 LLM-jp 32b": pick("llm-jp-4-32b"),
    "🐉 Qwen 27b":   pick("qwen"),
    "💎 Gemma 31b":  pick("gemma"),
}
SUBJECT_KEYS = list(VOICES.keys())
print("選手入場:")
for n, m in VOICES.items():
    print(f"  {n:18s} → {m}")


## 2. 集計用ユーティリティ

全競技で共通の **順位 → ポイント変換** と **記録ストア**。


In [ ]:
# 全競技の結果を蓄積するスコアボード
# scoreboard[competition][voice] = {"score": float, "raw_metric": value, "rank": int, "points": int}
scoreboard = defaultdict(dict)
# レイテンシも別に保管 (速度競技用)
latencies = defaultdict(list)  # voice -> [seconds, ...]
# 全競技の Borda 点を最後に合算
total_points = defaultdict(float)
medal_counts = defaultdict(lambda: {"🥇": 0, "🥈": 0, "🥉": 0, "🍀": 0})

def assign_ranks_and_points(competition, metric_dict, higher_is_better=True):
    """metric_dict: {voice: numeric_score}
    1位=4点, 2位=3点, 3位=2点, 4位=1点 (同点は平均)
    """
    items = sorted(metric_dict.items(),
                   key=lambda x: (-x[1] if higher_is_better else x[1]))
    # 同点処理: 同じスコアは同点
    points_table = [4, 3, 2, 1]
    medals = ["🥇", "🥈", "🥉", "🍀"]
    i = 0
    while i < len(items):
        j = i
        while j < len(items) and items[j][1] == items[i][1]:
            j += 1
        # i..j-1 が同点
        avg_pt = sum(points_table[i:j]) / (j - i)
        for k in range(i, j):
            voice, val = items[k]
            scoreboard[competition][voice] = {
                "metric": val,
                "rank": i + 1,
                "points": avg_pt,
                "medal": medals[i],
            }
            total_points[voice] += avg_pt
            medal_counts[voice][medals[i]] += 1
        i = j
    return items

def display_competition_result(competition_name, emoji, items, metric_label):
    rows = [f"### {emoji} {competition_name} 結果\n"]
    rows.append(f"| 順位 | 選手 | {metric_label} | 競技ポイント |")
    rows.append("|---|---|---|---|")
    for voice, val in items:
        entry = scoreboard[competition_name][voice]
        rows.append(f"| {entry['medal']} {entry['rank']} | {voice} | {val:.3f} | {entry['points']:.1f} |")
    display(Markdown("\n".join(rows)))


## 3. 🧠 競技 1: JCommonsenseQA (常識推論)

[JGLUE](https://github.com/yahoojapan/JGLUE) (山田/河原/柴田, NLP2022) の 5 択常識推論。Nejumi/llm-jp-eval にも採用されている代表的タスク。

サンプル質問 (公開セットより):
> 質問: 主に子ども向けのもので、イラストのついた物語が書かれているものはどれ？
> 0: 世界, 1: 写真集, 2: 絵本, 3: 論文, 4: 図鑑 → 正解 2

`jaster` 形式 (Nejumi で使われている標準 prompt) で問う。


In [ ]:
N_JCQA = 12  # ★ 増やすほど厳密な評価。Nejumi は dev 全数 1119 問。

print("📥 JCommonsenseQA をロード中...")
ds_jcqa = load_dataset("sbintuitions/JCommonsenseQA", split="validation")
random.seed(42)
indices = random.sample(range(len(ds_jcqa)), N_JCQA)
samples_jcqa = [ds_jcqa[i] for i in indices]
print(f"✓ {N_JCQA} 問サンプリング完了 (全 {len(ds_jcqa)} 問中)")

JCQA_PROMPT = """質問と回答の選択肢を入力として受け取り、選択肢から回答を選択してください。なお、回答は選択肢の番号 (例: 0) でするものとします。回答となる数値を1つだけ返し、他には何も含めないことを厳守してください。

質問: {q}
選択肢: 0.{c0}, 1.{c1}, 2.{c2}, 3.{c3}, 4.{c4}

回答:"""

def parse_jcqa(raw):
    if not raw: return -1
    # 全角→半角
    s = raw.translate(str.maketrans("０１２３４", "01234"))
    # 末尾近くから数字を探す (思考が前にあって最後に答えがある場合に強い)
    m = re.findall(r'[0-4]', s)
    return int(m[-1]) if m else -1

print(f"\n🧠 競技 1 開始 — {N_JCQA} 問 × {len(VOICES)} モデル = {N_JCQA * len(VOICES)} 呼び出し\n")
correct_jcqa = {v: 0 for v in VOICES}
wrong_examples_jcqa = []  # for highlights
right_examples_jcqa = []

for qi, item in enumerate(samples_jcqa):
    truth = item["label"]
    prompt = JCQA_PROMPT.format(
        q=item["question"],
        c0=item["choice0"], c1=item["choice1"], c2=item["choice2"],
        c3=item["choice3"], c4=item["choice4"],
    )
    print(f"  Q{qi+1:02d}: {item['question'][:40]:40s} (正解: {truth})", flush=True)
    answers = {}
    for voice, model in VOICES.items():
        raw, dt, _ = chat(model, prompt, max_tokens=1500)
        ans = parse_jcqa(raw)
        latencies[f"JCQA/{voice}"].append(dt)
        answers[voice] = ans
        mark = "✓" if ans == truth else "✗"
        print(f"    {voice:18s} {mark} {ans}  ({dt:.1f}s)")
        if ans == truth:
            correct_jcqa[voice] += 1
    # 興味深い例を保存
    if all(a != truth for a in answers.values()):
        wrong_examples_jcqa.append((item, answers))
    elif sum(1 for a in answers.values() if a == truth) == 1:
        # 1モデルだけ正解 = 興味深い
        right_examples_jcqa.append((item, answers))

acc_jcqa = {v: correct_jcqa[v] / N_JCQA for v in VOICES}
items_jcqa = assign_ranks_and_points("JCQA", acc_jcqa)
display_competition_result("JCommonsenseQA (常識推論)", "🧠", items_jcqa, "正答率")


## 4. 📚 競技 2: JMMLU (学術知識)

[JMMLU](https://huggingface.co/datasets/nlp-waseda/JMMLU) (尹/河原, 2024) は MMLU の日本語版 + 日本独自科目。
56 科目 7,536 問のうち、ここでは **4 科目から少数ずつサンプル** して学術知識の幅広さを測る。

> ⚠️ JMMLU のライセンスは CC BY-NC-ND 4.0 (研究・評価目的のみ、商用不可)


In [ ]:
N_JMMLU_PER_SUBJECT = 3  # ★ 1 科目あたり。Nejumi は科目ごと 5 問 × 56 科目 = 280 問
SUBJECTS = [
    "japanese_history",        # 日本史 (日本独自科目)
    "elementary_mathematics",  # 算数 (基礎)
    "professional_medicine",   # 医療 (専門知識)
    "philosophy",              # 哲学 (人文)
]

print("📥 JMMLU をロード中 (初回は数十秒)...")
# nlp-waseda/JMMLU は loading script を使う → trust_remote_code=True
ds_jmmlu_by_subject = {}
for subj in SUBJECTS:
    try:
        ds_jmmlu_by_subject[subj] = load_dataset(
            "nlp-waseda/JMMLU", subj, split="test", trust_remote_code=True
        )
        print(f"  ✓ {subj}: {len(ds_jmmlu_by_subject[subj])} 問")
    except Exception as e:
        print(f"  ✗ {subj}: load 失敗 ({type(e).__name__})")

JMMLU_PROMPT = """以下は{subject}に関する四択問題です。最も適切な選択肢の記号 (A, B, C, D) を1文字だけ返してください。他には何も含めないでください。

問題: {q}
A. {a}
B. {b}
C. {c}
D. {d}

回答:"""

JMMLU_SUBJECT_LABEL = {
    "japanese_history": "日本史",
    "elementary_mathematics": "初等数学",
    "professional_medicine": "専門医学",
    "philosophy": "哲学",
}

def parse_jmmlu(raw):
    if not raw: return None
    s = raw.translate(str.maketrans("ＡＢＣＤａｂｃｄ", "ABCDABCD"))
    m = re.findall(r'[ABCD]', s)
    return m[-1] if m else None

# サンプルを集める
random.seed(43)
samples_jmmlu = []
for subj in SUBJECTS:
    if subj not in ds_jmmlu_by_subject: continue
    ds = ds_jmmlu_by_subject[subj]
    idxs = random.sample(range(len(ds)), min(N_JMMLU_PER_SUBJECT, len(ds)))
    for i in idxs:
        samples_jmmlu.append((subj, ds[i]))
random.shuffle(samples_jmmlu)
N_JMMLU = len(samples_jmmlu)
print(f"\n📚 競技 2 開始 — {N_JMMLU} 問 × {len(VOICES)} = {N_JMMLU * len(VOICES)} 呼び出し\n")

correct_jmmlu = {v: 0 for v in VOICES}
correct_jmmlu_by_subject = {v: defaultdict(int) for v in VOICES}
total_jmmlu_by_subject = defaultdict(int)
wrong_examples_jmmlu = []

for qi, (subj, item) in enumerate(samples_jmmlu):
    truth = item["answer"]
    total_jmmlu_by_subject[subj] += 1
    prompt = JMMLU_PROMPT.format(
        subject=JMMLU_SUBJECT_LABEL[subj],
        q=item["question"], a=item["A"], b=item["B"], c=item["C"], d=item["D"],
    )
    print(f"  Q{qi+1:02d} [{JMMLU_SUBJECT_LABEL[subj]:6s}]: {item['question'][:30]:30s} (正解: {truth})")
    answers = {}
    for voice, model in VOICES.items():
        raw, dt, _ = chat(model, prompt, max_tokens=1500)
        ans = parse_jmmlu(raw)
        latencies[f"JMMLU/{voice}"].append(dt)
        answers[voice] = ans
        mark = "✓" if ans == truth else "✗"
        print(f"    {voice:18s} {mark} {ans}  ({dt:.1f}s)")
        if ans == truth:
            correct_jmmlu[voice] += 1
            correct_jmmlu_by_subject[voice][subj] += 1
    if all(a != truth for a in answers.values()):
        wrong_examples_jmmlu.append((subj, item, answers))

acc_jmmlu = {v: correct_jmmlu[v] / N_JMMLU for v in VOICES}
items_jmmlu = assign_ranks_and_points("JMMLU", acc_jmmlu)
display_competition_result("JMMLU (学術知識)", "📚", items_jmmlu, "正答率")

# 科目別ヒートマップ
rows = ["\n#### 📊 科目別正答率"]
rows.append("| 選手 | " + " | ".join(JMMLU_SUBJECT_LABEL[s] for s in SUBJECTS) + " |")
rows.append("|" + "---|" * (len(SUBJECTS) + 1))
for voice in VOICES:
    cells_r = [voice]
    for subj in SUBJECTS:
        c = correct_jmmlu_by_subject[voice][subj]
        t = total_jmmlu_by_subject[subj]
        cells_r.append(f"{c}/{t}" if t else "—")
    rows.append("| " + " | ".join(cells_r) + " |")
display(Markdown("\n".join(rows)))


## 5. ➗ 競技 3: MGSM-ja (数学・連鎖思考)

[MGSM](https://github.com/google-research/url-nlp/tree/main/mgsm) (Shi+, 2022) は GSM8K (小学校〜中学校レベルの数学文章題) を人手で 10 言語に翻訳した多言語ベンチマーク。
日本語subset (`ja`) は test 250 問。

ここでは **Chain-of-Thought (CoT) を促す prompt** で解かせる。「Let's think step by step」 (日本語: 「順を追って考えましょう」) は thinking モデルでは効果が薄いが、ベンチマークの公平性のため標準形式に従う。


In [ ]:
N_MGSM = 8  # ★ Nejumi は 250 問全数。8 問は短時間で 4 モデル比較できる程度。

print("📥 MGSM-ja をロード中...")
ds_mgsm = load_dataset("juletxara/mgsm", "ja", split="test")
random.seed(44)
samples_mgsm = random.sample(list(ds_mgsm), N_MGSM)
print(f"✓ {N_MGSM} 問サンプリング (全 {len(ds_mgsm)} 問中)")

MGSM_PROMPT = """次の数学の問題を、順を追って考えて解いてください。最後の行に「答え: 数値」の形式で答えを書いてください。

問題: {q}

順を追った解答:"""

# 答えの数字を抽出 (後ろに「答え:」がある場合はそれ優先、無ければ最後の数値)
ANS_PAT = re.compile(r'答え\s*[:：は]\s*(-?[0-9,]+(?:\.[0-9]+)?)')
NUM_PAT = re.compile(r'-?[0-9,]+(?:\.[0-9]+)?')
def parse_mgsm(raw):
    if not raw: return None
    s = raw.translate(str.maketrans("０１２３４５６７８９．，", "0123456789.,"))
    # 「答え: N」が末尾近くにあればそれを採用
    matches = list(ANS_PAT.finditer(s))
    if matches:
        token = matches[-1].group(1)
    else:
        # 最後の数値を採用
        toks = NUM_PAT.findall(s)
        if not toks: return None
        token = toks[-1]
    token = token.replace(",", "")
    try:
        v = float(token)
        return int(v) if v == int(v) else v
    except ValueError:
        return None

print(f"\n➗ 競技 3 開始 — {N_MGSM} 問 × {len(VOICES)} = {N_MGSM * len(VOICES)} 呼び出し\n")
correct_mgsm = {v: 0 for v in VOICES}
wrong_examples_mgsm = []
right_examples_mgsm = []

for qi, item in enumerate(samples_mgsm):
    truth = item["answer_number"]
    prompt = MGSM_PROMPT.format(q=item["question"])
    print(f"  Q{qi+1:02d}: {item['question'][:50]:50s} (正解: {truth})")
    answers = {}
    for voice, model in VOICES.items():
        raw, dt, _ = chat(model, prompt, max_tokens=4000)  # CoT は長め
        ans = parse_mgsm(raw)
        latencies[f"MGSM/{voice}"].append(dt)
        answers[voice] = (ans, raw)
        mark = "✓" if ans == truth else "✗"
        print(f"    {voice:18s} {mark} {ans}  ({dt:.1f}s)")
        if ans == truth:
            correct_mgsm[voice] += 1
    if all(a[0] != truth for a in answers.values()):
        wrong_examples_mgsm.append((item, answers))
    elif sum(1 for a in answers.values() if a[0] == truth) == 1:
        right_examples_mgsm.append((item, answers))

acc_mgsm = {v: correct_mgsm[v] / N_MGSM for v in VOICES}
items_mgsm = assign_ranks_and_points("MGSM", acc_mgsm)
display_competition_result("MGSM-ja (数学CoT)", "➗", items_mgsm, "正答率")


## 6. 💬 競技 4: 自由対話 (LLM-as-judge / Borda count)

JMT-Bench (Rakuda / 日本語MT-Bench) では強い審判モデル (GPT-4o) に 1-10 点で採点させる。
ここでは **GPT-4o は使わず、4 モデル自身を匿名相互審査**させる (`03_haiku_battle.ipynb` と同じ手法、Constitutional AI 系研究で広く検証済み)。

プロンプトは **JaMT-Bench カテゴリ** から 1 つずつ抽出 (writing / reasoning / extraction)。


In [ ]:
N_PAIRWISE = 3  # ★ お題数。JaMT-Bench 本体は 80 問。

PROMPTS = [
    ("writing",
     "あなたは技術ブログの執筆者です。「AI と研究者の協働」というテーマで、200 字程度の導入文を書いてください。"),
    ("reasoning",
     "ある研究室で 5 人がそれぞれ別の言語を使っています。日本語, 英語, フランス語, ドイツ語, 中国語です。"
     "(1) 田中さんは英語が分からない (2) ドイツ語話者は田中さんの隣 (3) 鈴木さんはフランス語 "
     "(4) 中国語話者は鈴木さんの 2 つ隣 (5) 山田さんは日本語の隣に座っている "
     "田中さんが話す言語は何ですか。簡潔に推論過程と結論を述べてください。"),
    ("extraction",
     "次の研究室通信から、(a) 開催日、(b) 主題、(c) 招待講演者 を JSON で抽出してください。"
     "余計な説明は不要、JSON のみ出力すること。\n\n"
     "通信文: 「来る 6 月 14 日 (金)、月例研究会を本館 3F セミナー室にて開催します。"
     "今回の主題は『大規模言語モデルの数学的能力』、招待講演は東京大学の佐藤一郎先生です。"
     "皆様のご参加お待ちしております。」"),
]
PROMPTS = PROMPTS[:N_PAIRWISE]

print(f"💬 競技 4 開始 — 各お題で {len(VOICES)} 答案 + {len(VOICES)} 審査 = "
      f"{N_PAIRWISE * len(VOICES) * 2} 呼び出し\n")

# 答案収集
answers_by_prompt = []  # [(category, prompt, {voice: text})]
for pi, (cat, p) in enumerate(PROMPTS):
    print(f"  📝 お題 {pi+1} [{cat}]: {p[:50]}...")
    ans_dict = {}
    for voice, model in VOICES.items():
        raw, dt, _ = chat(model, p, max_tokens=3000, temperature=0.7)
        ans_dict[voice] = raw
        latencies[f"DIALOG/{voice}"].append(dt)
        print(f"    {voice:18s} → {len(raw):4d}字 ({dt:.1f}s)")
    answers_by_prompt.append((cat, p, ans_dict))

# 各お題を匿名化 A/B/C/D シャッフル + 相互審査
JUDGE_PROMPT = """以下は同じ質問に対する 4 つの回答 A, B, C, D です。
質問への適切さ・正確さ・明瞭さの観点で 1 位から 4 位まで順位を付けてください。

質問: {q}

回答 A: {a}

回答 B: {b}

回答 C: {c}

回答 D: {d}

回答は厳密に以下の形式で出力してください (他は一切書かないこと):
1位: X
2位: X
3位: X
4位: X
(X には A B C D のいずれか1文字を入れる)
"""
RANK_PAT = re.compile(r'([1-4１-４一二三四])\s*位\s*[:：]?\s*([ABCDＡＢＣＤ])')
ZEN2HAN = str.maketrans("１２３４一二三四ＡＢＣＤ", "12341234ABCD")
def parse_ranks(text):
    s = text.translate(ZEN2HAN)
    out = {}
    for m in RANK_PAT.finditer(s):
        r = int(m.group(1))
        if r not in out:
            out[r] = m.group(2)[0]
    return out

# 各お題でランキング → Borda 計算
dialog_scores = {v: 0.0 for v in VOICES}  # 全プロンプトでの平均ポイント

for pi, (cat, p, ans_dict) in enumerate(answers_by_prompt):
    voices_list = list(ans_dict.keys())
    random.seed(100 + pi)
    order = list(range(4))
    random.shuffle(order)
    labels = ["A", "B", "C", "D"]
    label_to_voice = {labels[i]: voices_list[order[i]] for i in range(4)}
    
    print(f"\n  🧑‍⚖️ お題 {pi+1} の審査 (匿名化済み)")
    judge_prompt_str = JUDGE_PROMPT.format(
        q=p,
        a=ans_dict[label_to_voice["A"]][:1500],
        b=ans_dict[label_to_voice["B"]][:1500],
        c=ans_dict[label_to_voice["C"]][:1500],
        d=ans_dict[label_to_voice["D"]][:1500],
    )
    # 各モデルが判定。自分の答案も混ざるが匿名なので原則OK
    prompt_borda = {v: [] for v in voices_list}  # voice -> list of points from each judge
    for judge_voice, judge_model in VOICES.items():
        raw, dt, _ = chat(judge_model, judge_prompt_str, max_tokens=2000, temperature=0.0)
        ranks = parse_ranks(raw)
        latencies[f"JUDGE/{judge_voice}"].append(dt)
        if len(ranks) >= 3:  # 部分パースでも採用
            # 順位 r が出ているラベルに ( 5 - r ) 点 (1位=4, 4位=1)
            for r, lab in ranks.items():
                if lab in label_to_voice:
                    voice = label_to_voice[lab]
                    prompt_borda[voice].append(5 - r)
            sorted_view = [f"{r}位={ranks.get(r, '?')}" for r in [1,2,3,4]]
            print(f"    判定 by {judge_voice}: {' '.join(sorted_view)}")
        else:
            print(f"    判定 by {judge_voice}: ⚠️ パース失敗 (ranks={ranks})")
    # 各 voice の平均ポイント (4人の judge から)
    for voice in voices_list:
        if prompt_borda[voice]:
            dialog_scores[voice] += statistics.mean(prompt_borda[voice])

# 平均化 (お題数で割る) して 4-1 スケールに収まるよう正規化済み
dialog_scores_norm = {v: dialog_scores[v] / N_PAIRWISE for v in VOICES}
items_dialog = assign_ranks_and_points("DIALOG", dialog_scores_norm)
display_competition_result("自由対話 (LLM-as-judge Borda)", "💬", items_dialog, "平均ポイント")


## 7. 🏆 表彰式

全 4 競技の Borda 点を合算 → 総合ランキング。
ASCII 表彰台 + メダル数も並記。


In [ ]:
overall = sorted(total_points.items(), key=lambda x: -x[1])
print("=" * 60)
print(" 🏟️  4 モデル日本語LLMベンチマーク選手権 最終結果")
print("=" * 60)
print()

# ASCII 表彰台
def podium_box(rank_emoji, voice, pts):
    # 短く整形
    name = voice
    return f"{rank_emoji} {name} ({pts:.1f}pt)"

if len(overall) >= 3:
    p1 = podium_box("🥇", *overall[0])
    p2 = podium_box("🥈", *overall[1])
    p3 = podium_box("🥉", *overall[2])
    print(f"               {p1}")
    print( "              ┌────────┐")
    print(f"   {p2:30s}│         │")
    print(f"  ┌────────┐ │   1位  │ ┌────────┐  {p3}" if len(overall) >= 3 else "")
    print( "  │   2位  │ │         │ │   3位  │")
    print( "  │         │ │         │ │         │")
    print( "  └────────┘ └────────┘ └────────┘")
    print()
    if len(overall) >= 4:
        print(f"  入賞 🍀: {podium_box('', *overall[3])}")

# メダル + 競技別内訳
rows = ["", "### 📊 総合ランキング"]
rows.append("| 順位 | 選手 | 総合点 | 🥇 | 🥈 | 🥉 | 🍀 | JCQA | JMMLU | MGSM | DIALOG |")
rows.append("|" + "---|" * 11)
for i, (voice, pts) in enumerate(overall):
    mc = medal_counts[voice]
    cells_r = [
        f"{i+1}",
        voice,
        f"{pts:.1f}",
        str(mc["🥇"]), str(mc["🥈"]), str(mc["🥉"]), str(mc["🍀"]),
    ]
    for comp in ["JCQA", "JMMLU", "MGSM", "DIALOG"]:
        entry = scoreboard[comp].get(voice, {})
        rank = entry.get("rank", "-")
        metric = entry.get("metric", None)
        if metric is None:
            cells_r.append("—")
        elif comp == "DIALOG":
            cells_r.append(f"{entry['medal']}{rank} ({metric:.2f})")
        else:
            cells_r.append(f"{entry['medal']}{rank} ({metric:.0%})")
    rows.append("| " + " | ".join(cells_r) + " |")
display(Markdown("\n".join(rows)))


## 8. ⚡ 速度 vs 精度: 「コスパ」の観点

各モデルの平均レイテンシと正答率を並べる。
**正答 1 つあたり何秒か** で「実用効率」を見る (推論コスト換算の代用)。


In [ ]:
import math

print("⚡ 速度・精度分析\n")

rows = ["### ⚡ レイテンシ & コスパ", "", 
        "| 選手 | JCQA平均 | JMMLU平均 | MGSM平均 | DIALOG平均 | 総合正答 | 1正答あたり秒 |",
        "|" + "---|" * 7]

for voice in VOICES:
    lat_jcqa = statistics.mean(latencies[f"JCQA/{voice}"]) if latencies[f"JCQA/{voice}"] else 0
    lat_jmmlu = statistics.mean(latencies[f"JMMLU/{voice}"]) if latencies[f"JMMLU/{voice}"] else 0
    lat_mgsm = statistics.mean(latencies[f"MGSM/{voice}"]) if latencies[f"MGSM/{voice}"] else 0
    lat_dialog = statistics.mean(latencies[f"DIALOG/{voice}"]) if latencies[f"DIALOG/{voice}"] else 0
    
    # JCQA + JMMLU + MGSM で「総合正答数」と「総合時間」を出す
    total_correct = (correct_jcqa[voice] + correct_jmmlu[voice] + correct_mgsm[voice])
    total_time = (sum(latencies[f"JCQA/{voice}"]) + sum(latencies[f"JMMLU/{voice}"]) 
                  + sum(latencies[f"MGSM/{voice}"]))
    sec_per_correct = (total_time / total_correct) if total_correct > 0 else float("inf")
    rows.append(
        f"| {voice} | {lat_jcqa:.1f}s | {lat_jmmlu:.1f}s | {lat_mgsm:.1f}s | "
        f"{lat_dialog:.1f}s | {total_correct}/{N_JCQA + N_JMMLU + N_MGSM} | "
        f"{sec_per_correct:.1f}s |"
    )

display(Markdown("\n".join(rows)))

# 観察コメント
fastest = min(VOICES, key=lambda v: statistics.mean(latencies[f"JCQA/{v}"]) if latencies[f"JCQA/{v}"] else float("inf"))
slowest = max(VOICES, key=lambda v: statistics.mean(latencies[f"JCQA/{v}"]) if latencies[f"JCQA/{v}"] else 0)
print(f"\n💨 最速 (JCQA): {fastest}")
print(f"🐢 最遅 (JCQA): {slowest}")
print()
print("注: thinking モデル (LLM-jp-4, Qwen3, Gemma-3) は recommendation reasoning に時間を要するため")
print("    一般に遅いが、難問になるほど正答率で挽回することがある。MGSM 結果と JCQA 結果を比較すると傾向が見える。")


## 9. 🎬 ハイライト & ローライト

興味深いケースを抜粋:
- **🌟 全員正解** → ベンチマーク的に簡単な問題 (省略)
- **🤯 全員不正解** → モデルに共通の弱点 (重要、列挙)
- **💎 1モデルだけ正解** → そのモデルの強み (列挙)


In [ ]:
display(Markdown("### 🤯 全員不正解 (全モデルの弱点)\n"))

# JCQA
if wrong_examples_jcqa:
    rows = ["#### JCQA 全滅例", ""]
    for item, answers in wrong_examples_jcqa[:3]:
        rows.append(f"- **問題**: {item['question']}")
        choices = [item[f'choice{i}'] for i in range(5)]
        rows.append(f"  - 選択肢: " + ", ".join(f"{i}.{c}" for i, c in enumerate(choices)))
        rows.append(f"  - **正解**: {item['label']} ({choices[item['label']]})")
        rows.append(f"  - 各モデル回答: " + ", ".join(f"{v}={ans}" for v, ans in answers.items()))
        rows.append("")
    display(Markdown("\n".join(rows)))

# MGSM
if wrong_examples_mgsm:
    rows = ["#### MGSM 全滅例", ""]
    for item, answers in wrong_examples_mgsm[:2]:
        rows.append(f"- **問題**: {item['question']}")
        rows.append(f"  - **正解**: {item['answer_number']}")
        rows.append(f"  - 各モデル回答: " + ", ".join(f"{v}={a[0]}" for v, a in answers.items()))
        rows.append("")
    display(Markdown("\n".join(rows)))

if wrong_examples_jmmlu:
    rows = ["#### JMMLU 全滅例", ""]
    for subj, item, answers in wrong_examples_jmmlu[:2]:
        rows.append(f"- **[{JMMLU_SUBJECT_LABEL[subj]}]**: {item['question']}")
        rows.append(f"  - A. {item['A']}, B. {item['B']}, C. {item['C']}, D. {item['D']}")
        rows.append(f"  - **正解**: {item['answer']}")
        rows.append(f"  - 各モデル回答: " + ", ".join(f"{v}={ans}" for v, ans in answers.items()))
        rows.append("")
    display(Markdown("\n".join(rows)))

display(Markdown("\n### 💎 1モデルだけ正解 (そのモデルの強み)\n"))

if right_examples_jcqa:
    rows = ["#### JCQA 一抜け例", ""]
    for item, answers in right_examples_jcqa[:3]:
        truth = item["label"]
        winner = next(v for v, a in answers.items() if a == truth)
        rows.append(f"- **問題**: {item['question']}")
        rows.append(f"  - 正解: {truth} → **{winner} だけ正解** 🎯")
        rows.append(f"  - 各モデル回答: " + ", ".join(f"{v}={ans}" for v, ans in answers.items()))
        rows.append("")
    display(Markdown("\n".join(rows)))

if right_examples_mgsm:
    rows = ["#### MGSM 一抜け例", ""]
    for item, answers in right_examples_mgsm[:2]:
        truth = item["answer_number"]
        winner = next(v for v, a in answers.items() if a[0] == truth)
        rows.append(f"- **問題**: {item['question'][:80]}...")
        rows.append(f"  - 正解: {truth} → **{winner} だけ正解** 🎯")
        rows.append(f"  - 各モデル回答: " + ", ".join(f"{v}={a[0]}" for v, a in answers.items()))
        rows.append("")
    display(Markdown("\n".join(rows)))


## 10. 📈 Nejumi Leaderboard との比較

[Nejumi LLM Leaderboard](https://wandb.ai/wandb-japan/llm-leaderboard) は Weights & Biases Japan が運営する公式リーダーボードで、フルサイズ評価を行っている。
ここで実走したのは数十問の **小サンプル評価**なので、95% 信頼区間は広い (Wilson 区間で ±15% 前後)。

**期待される傾向** (公開情報ベース):
- LLM-jp-4 32B-A3B は LLM-jp-4 8B より JMMLU で +10 ポイント前後高い
- Qwen3-VL-27B-A3B (thinking) は MGSM で日本語モデル群の中で上位
- Gemma3-27B は対話応答 (JaMT-Bench style) が強い傾向

実走結果と公開スコアが大きく乖離する場合の解釈:
1. **サンプルが少ない** (本実装 N=8〜12) → 偶発的ばらつき
2. **prompt format の違い** → Nejumi は jaster 形式 (本実装は近似)
3. **temperature の違い** → 本実装は 0.0 だが、Nejumi は通常 greedy
4. **thinking budget の違い** → Playground の non-streaming パスで reasoning が落ちる現象 (`01_api_basics.ipynb` 参照)

サンプル数を増やして再走させる場合:
```python
N_JCQA = 50; N_JMMLU_PER_SUBJECT = 10; N_MGSM = 30; N_PAIRWISE = 8
```
これで全モデル合計 ~5000 API 呼び出し、所要 6〜10 時間程度。


In [ ]:
# 95% Wilson 信頼区間 (簡易) で実走結果を提示
import math
def wilson(p, n, z=1.96):
    if n == 0: return (0, 0)
    denom = 1 + z*z / n
    centre = (p + z*z/(2*n)) / denom
    half = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return (max(0, centre - half), min(1, centre + half))

rows = ["### 📊 実走結果 ± 95% 信頼区間", ""]
rows.append("| 選手 | JCQA | JMMLU | MGSM |")
rows.append("|---|---|---|---|")
for voice in VOICES:
    parts = [voice]
    for comp, n, correct in [("JCQA", N_JCQA, correct_jcqa[voice]),
                              ("JMMLU", N_JMMLU, correct_jmmlu[voice]),
                              ("MGSM", N_MGSM, correct_mgsm[voice])]:
        p = correct / n if n else 0
        lo, hi = wilson(p, n)
        parts.append(f"{p:.0%} ({lo:.0%}-{hi:.0%})")
    rows.append("| " + " | ".join(parts) + " |")
display(Markdown("\n".join(rows)))

display(Markdown(
    "**読み方**: 信頼区間が広い (例: 30%-70%) のは、サンプル数 N が小さいため。"
    "N を 50〜100 に増やすと区間が ±10% 以下に収束する。"
    "Nejumi のフルサイズ評価では N=数百〜千なので、点推定がほぼ確定値として比較される。"
))


## 11. おまけ — 拡張のアイデア

- **`N_*` を増やしてフル走**: 各タスク 50〜100 問にすると、Wilson 信頼区間が ±10% 以下に収まる
- **追加ベンチマーク**:
  - `JTruthfulQA` (幻覚評価) — `nlp-waseda/JTruthfulQA`
  - `JHumanEval` (コード生成) — `kogi-jwu/jhumaneval`
  - `Jamp` (時間推論) — `tomo-makes/jamp`
  - `NIILC` (国立情報学研究所の質問応答) — `kumiokmd/niilc`
- **形式の sensitivity 検証**: Nejumi 流に「同じ問題を A〜D 記号でなく数字で問う」「正解を選ばせず誤りを選ばせる」など prompt 摂動でロバストネスを測る
- **CoT vs no-CoT**: MGSM を CoT 無効で再走させて差分を見る (thinking model なら差が小さいはず)
- **Generator-Verifier-Reviser パイプラインへの応用**: 各モデルの強み・弱みが見えたら、Verifier に使うべきモデルが決まる (`02_gvr_pipeline.ipynb` に活かす)

## 参考文献

- 栗原健太郎, 河原大輔, 柴田知秀. **JGLUE: 日本語言語理解ベンチマーク**. 言語処理学会第28回年次大会, 2022.
- Z. Yin, et al. **JMMLU: Japanese Massive Multitask Language Understanding Benchmark**. 2024. https://huggingface.co/datasets/nlp-waseda/JMMLU
- F. Shi, M. Suzgun, et al. **Language Models are Multilingual Chain-of-Thought Reasoners**. arXiv:2210.03057, 2022.
- Z. Zheng, et al. **Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena**. NeurIPS 2023.
- W&B Japan. **Nejumi LLM Leaderboard 4**. https://wandb.ai/wandb-japan/llm-leaderboard
- LLM-jp. **llm-jp-eval**. https://github.com/llm-jp/llm-jp-eval
